In [1]:
raw_text = (
    "제1장 총칙: 본 규정은 컴퓨터공학과의 실습실 이용 규정을 명시합니다.\n\n"
    "제2장 이용 시간: 평일은 오전 9시부터 오후 10시까지 개방합니다. 주말은 미개방합니다."
)
file_info = {"source": "cs_lab_policy.pdf", "department": "컴퓨터공학과", "version": "v1.2"}

In [2]:
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [6]:
doc = Document(page_content=raw_text, metadata=file_info)

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=60,
    chunk_overlap=0
)

docs = text_splitter.split_documents([doc])

for i, d in enumerate(docs, 1):
    print(f"[{i}번째] {d}")
    print("-" * 50)

[1번째] page_content='제1장 총칙: 본 규정은 컴퓨터공학과의 실습실 이용 규정을 명시합니다.' metadata={'source': 'cs_lab_policy.pdf', 'department': '컴퓨터공학과', 'version': 'v1.2'}
--------------------------------------------------
[2번째] page_content='제2장 이용 시간: 평일은 오전 9시부터 오후 10시까지 개방합니다. 주말은 미개방합니다.' metadata={'source': 'cs_lab_policy.pdf', 'department': '컴퓨터공학과', 'version': 'v1.2'}
--------------------------------------------------


In [7]:
r_doc = docs[1]
r_doc

Document(metadata={'source': 'cs_lab_policy.pdf', 'department': '컴퓨터공학과', 'version': 'v1.2'}, page_content='제2장 이용 시간: 평일은 오전 9시부터 오후 10시까지 개방합니다. 주말은 미개방합니다.')

In [9]:
from langchain.chat_models import init_chat_model
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser


model = init_chat_model(
    model_provider="openai",
    model="gpt-6-luna",
    reasoning_effort="low"
)

parser = StrOutputParser()

prompt = ChatPromptTemplate.from_messages([
    ("system", (
        "제공된 [참고 문서]에만 근거하여 답변하세요. [참고 문서]에 없는 내용은 '잘 모르겠습니다.' 라고 답변하세요.\n"
        "답변 끝에는 반드시 [출처: 학과(파일이름)] 형식으로 근거를 명시하세요.\n\n"
        "[참고 문서]:\n{context}\n\n"
        "[문서 메타데이터]: 학과={dept}, 파일이름={source}"
    )),
    ("user", "{question}")
])

chain = prompt | model | parser
user_query = "실습실은 주말에 이용할 수 있나요?"

response = chain.invoke({
    "context": r_doc.page_content,
    "dept": r_doc.metadata['department'],
    "source": r_doc.metadata['source'],
    "question": user_query
})

print(response)

주말에는 개방하지 않습니다. [출처: 컴퓨터공학과(cs_lab_policy.pdf)]
